# Medical Classification

In [ ]:
! pip install pillow==9.2.0 scikit-image==0.19.2

## Preprocessing

In [ ]:
import cv2
import matplotlib.pyplot
import numpy
import os
import pandas
import PIL
import PIL.ImageFile
import seaborn
import skimage
import time

root_path = '/content/drive/MyDrive/Projects/Siraj Raval/Medical Classification/medical_classification'

class EDA:
    """
    Exploratory Data Analysis
    """
    def __init__(self):
        dict_labels = {
            0: "No DR",
            1: "Mild",
            2: "Moderate",
            3: "Severe",
            4: "Proliferative DR"}

    def __call__(self):
        labels = pandas.read_csv("labels/trainLabels.csv")
        plot_classification_frequency(labels, "level", "Retinopathy_vs_Frequency_All")
        plot_classification_frequency(labels, "level", "Retinopathy_vs_Frequency_Binary", True)

    def change_labels(df, category):
        """
        Changes the labels for a binary classification.
        Either the person has a degree of retinopathy, or they don't.
        Parameters
            df: Pandas DataFrame of the image name and labels
            category: column of the labels
        Return
            Column containing a binary classification of 0 or 1
        """
        return [1 if l > 0 else 0 for l in df[category]]

    def plot_classification_frequency(df, category, file_name, convert_labels=False):
        """
        Plots the frequency at which labels occur.
        Parameters
            df: Pandas DataFrame of the image name and labels
            category: category of labels, from 0 to 4
            file_name: file name of the image
            convert_labels: argument specified for converting to binary classification
        Return
            None
        """
        if convert_labels == True:
            labels['level'] = change_labels(labels, 'level')
        seaborn.set(style="whitegrid", color_codes=True)
        seaborn.countplot(x=category, data=labels)
        pyplot.title('Retinopathy vs Frequency')
        pyplot.savefig(file_name)
        return

In [ ]:
EDA()()

In [ ]:
class CropAndResizeImages:
    """
    TODO: docstring
    """
    PIL.ImageFile.LOAD_TRUNCATED_IMAGES = True

    def crop_and_resize_images(path, new_path, cropx, cropy, img_size):
        """
        Crops, resizes, and stores all images from a directory in a new directory.

        Parameters
            path: Path where the current, unscaled images are contained.
            new_path: Path to save the resized images.
            img_size: New size for the rescaled images.

        Return
            none
        """
        if not os.path.exists(new_path):
            os.makedirs(new_path)
        dirs = [l for l in os.listdir(path)]
        total = 0
        for item in dirs:
            img = skimage.io.imread(path + item)
            y, x, channel = img.shape
            startx = x // 2 - (cropx // 2)
            starty = y // 2 - (cropy // 2)
            img = img[starty:starty+cropy, startx:startx+cropx]
            img = skimage.transform.resize(img, (img_size, img_size))
            skimage.io.imsave(str(new_path + item), img)
            total += 1
            print('saving:', item, total)

In [ ]:
CropAndResizeImages.crop_and_resize_images(
    path=os.path.join(root_path, 'data/train/'),
    new_path=os.path.join(root_path, 'data/train-resized-256/'),
    cropx=1800, cropy=1800, img_size=256)

In [ ]:
CropAndResizeImages.crop_and_resize_images(
    path=os.path.join(root_path, 'data/test/'),
    new_path=os.path.join(root_path, 'data/test-resized-256/'),
    cropx=1800, cropy=1800, img_size=256)

In [ ]:
class PreprocessImages:
    """
    TODO: docstring
    """
    def __call__(self):
        """
        TODO: docstring
        """
        trainLabels = pandas.read_csv(os.path.join(root_path, 'data/trainLabels.csv'))
        trainLabels['image'] = [i + '.jpeg' for i in trainLabels['image']]
        trainLabels['black'] = numpy.nan
        trainLabels['black'] = self.find_black_images(os.path.join(
            root_path, 'data/train-resized-256/'), trainLabels)
        trainLabels = trainLabels.loc[trainLabels['black'] == 0]
        trainLabels.to_csv(os.path.join(
            root_path, 'data/trainLabels-master.csv'), index=False, header=True)
        print('completed')

    def find_black_images(self, file_path, df):
        """
        Creates a column of images that are not black (numpy.mean(img) != 0)

        Parameters
            file_path: file_path to the images to be analyzed.
            df: Pandas DataFrame that includes all labeled image names.
            column: column in DataFrame query is evaluated against.

        Return
            Column indicating if the photo is pitch black or not.
        """
        lst_imgs = [l for l in df['image']]
        return [1 if numpy.mean(numpy.array(
            PIL.Image.open(file_path + img))) == 0 else 0 for img in lst_imgs]
    
    def rename_images(self, src_dir, new_prefix):
        """
        TODO: docstring
        """
        for file_name in os.listdir(src_dir):
            os.rename(
                os.path.join(src_dir, file_name),
                os.path.join(src_dir, new_prefix + file_name))
            print(file_name + ' -> ' + new_prefix + file_name)

In [ ]:
PreprocessImages()()

In [ ]:
class RotateImages:
    """
    TODO: docstring
    """
    def __call__(self):
        """
        TODO: docstring
        """
        trainLabels = pandas.read_csv(os.path.join(
            root_path, 'data/trainLabels-master.csv'))
        trainLabels['image'] = trainLabels['image'].str.rstrip('.jpeg')
        trainLabels_no_DR = trainLabels[trainLabels['level'] == 0]
        trainLabels_DR = trainLabels[trainLabels['level'] >= 1]
        lst_imgs_no_DR = [i for i in trainLabels_no_DR['image']]
        lst_imgs_DR = [i for i in trainLabels_DR['image']]
        print('mirroring non-DR images')
        self.mirror_images(os.path.join(
            root_path, 'data/train-resized-256/'), 1, lst_imgs_no_DR)
        print('mirroring DR images')
        print('rotating 90 degrees')
        self.rotate_images(os.path.join(
            root_path, 'data/train-resized-256/'), 90, lst_imgs_DR)
        print('rotating 120 degrees')
        self.rotate_images(os.path.join(
            root_path, 'data/train-resized-256/'), 120, lst_imgs_DR)
        print('rotating 180 degrees')
        self.rotate_images(os.path.join(
            root_path, 'data/train-resized-256/'), 180, lst_imgs_DR)
        print('rotating 270 degrees')
        self.rotate_images(os.path.join(
            root_path, 'data/train-resized-256/'), 270, lst_imgs_DR)
        print('mirroring DR images')
        self.mirror_images(os.path.join(
            root_path, 'data/train-resized-256/'), 0, lst_imgs_DR)
        print('completed')

    def mirror_images(self, file_path, mirror_direction, lst_imgs):
        """
        Mirrors image left or right, based on criteria specified.

        Parameters
            file_path: file path to the folder containing images.
            mirror_direction: criteria for mirroring left or right.
            lst_imgs: list of image strings.

        Return
            None
        """
        for l in lst_imgs:
            img = cv2.imread(file_path + str(l) + '.jpeg')
            img = cv2.flip(img, 1)
            cv2.imwrite(file_path + str(l) + '_mir' + '.jpeg', img)

    def rotate_images(self, file_path, degrees_of_rotation, lst_imgs):
        """
        Rotates image based on a specified amount of degrees.

        Parameters
            file_path: file path to the folder containing images.
            degrees_of_rotation: Integer, specifying degrees to rotate the
            image. Set number from 1 to 360.
            lst_imgs: list of image strings.

        Return
            None
        """
        for l in lst_imgs:
            img = skimage.io.imread(file_path + str(l) + '.jpeg')
            img = skimage.transform.rotate(img, degrees_of_rotation)
            skimage.io.imsave(file_path + str(l) + '_' + str(degrees_of_rotation) + '.jpeg', img)

In [ ]:
RotateImages()()

In [ ]:
class ReconcileLabels:
    """
    TODO: docstring
    """
    def reconcile_labels():
        """
        TODO: docstring
        """
        trainLabels = pandas.read_csv(os.path.join(
            root_path, 'data/trainLabels-master.csv'))
        lst_imgs = [i for i in os.listdir(os.path.join(
            root_path, 'data/train-resized-256/')) if i != '.DS_Store']
        new_trainLabels = pandas.DataFrame({'image': lst_imgs})
        new_trainLabels['image2'] = new_trainLabels.image
        new_trainLabels['image2'] = new_trainLabels.loc[:, 'image2'].apply(
            lambda x: '_'.join(x.split('_')[0:2]))
        new_trainLabels['image2'] = new_trainLabels.loc[:, 'image2'].apply(
            lambda x: '_'.join(x.split('_')[0:2]).strip('.jpeg') + '.jpeg')
        new_trainLabels.columns = ['train_image_name', 'image']
        trainLabels = pandas.merge(
            trainLabels, new_trainLabels, how='outer', on='image')
        trainLabels.drop(['black'], axis=1, inplace=True)
        trainLabels = trainLabels.dropna()
        print(trainLabels.shape)
        print('writing CSV')
        trainLabels.to_csv(os.path.join(
            root_path, 'data/trainLabels-master_256_v2.csv'),
            index=False, header=True)

In [ ]:
ReconcileLabels.reconcile_labels()

In [ ]:
import cupy

In [ ]:
class ImageToArray:
    """
    TODO: docstring
    """
    def image_to_array():
        """
        TODO: docstring
        """
        labels = pandas.read_csv(os.path.join(
            root_path, 'data/trainLabels-master_256_v2.csv'))
        labels = labels[:10000]
        print('writing train array')
        lst_imgs = [l for l in labels['train_image_name']]
        X_train = cupy.array([cupy.array(PIL.Image.open(os.path.join(
            root_path, 'data/train-resized-256/') + img)) for img in lst_imgs])
        print(X_train.shape)
        print('saving train array')
        cupy.save(os.path.join(root_path, 'data/X_train_256_v2-10000.npy'), X_train)

In [ ]:
ImageToArray.image_to_array()

## Training

In [ ]:
import cupy
import numpy
import os
import pandas
import sklearn.model_selection
import sklearn.utils
import tensorflow

root_path = '/content/drive/MyDrive/Projects/Siraj Raval/Medical Classification/medical_classification'

class EyeNet:
    """
    TODO: docstring
    """
    def __init__(self):
        """
        TODO: docstring
        """
        cupy.random.seed(1337)
        self.channels = 3
        self.img_cols = 256
        self.img_rows = 256
        self.nb_classes = 5
        self.split_data(
            y_file_path=os.path.join(root_path, 'data/trainLabels-master_256_v2.csv'),
            X=os.path.join(root_path, 'data/X_train_256_v2-10000.npy'))
        self.reshape_data(self.img_rows, self.img_cols, self.channels, self.nb_classes)
        model = self.cnn_model(
            nb_filters=32, kernel_size=(4, 4), batch_size=512, nb_epoch=50)
        precision, recall, f1, cohen_kappa, quad_kappa = self.predict()
        print('Precision:', precision)
        print('Recall:', recall)
        print('F1:', f1)
        print('Cohen Kappa Score:', cohen_kappa)
        print('Quadratic Kappa:', quad_kappa)
        self.save_model(score=recall, model_name='DR_Class')

    def cnn_model(self, nb_filters, kernel_size, batch_size, nb_epoch):
        """
        Define and run the Convolutional Neural Network.

        Parameters
            nb_filters: initial number of filters
            kernel_size: initial size of kernel
            batch_size: batch size for the model
            nb_epoch: number of epochs

        Return
            Fitted CNN model
        """
        model = tensorflow.keras.models.Sequential()
        model.add(tensorflow.keras.layers.convolutional.Conv2D(
            nb_filters, (kernel_size[0], kernel_size[1]), padding='valid', strides=1,
            input_shape=(self.img_rows, self.img_cols, self.channels), activation='relu'))
        model.add(tensorflow.keras.layers.convolutional.Conv2D(
            nb_filters, (kernel_size[0], kernel_size[1]), activation='relu'))
        model.add(tensorflow.keras.layers.convolutional.Conv2D(
            nb_filters, (kernel_size[0], kernel_size[1]), activation='relu'))
        model.add(tensorflow.keras.layers.MaxPooling2D(pool_size=(8, 8)))
        model.add(tensorflow.keras.layers.Flatten())
        print('model flattened out to: ', model.output_shape)
        model.add(tensorflow.keras.layers.Dense(2048, activation='relu'))
        model.add(tensorflow.keras.layers.Dropout(0.25))
        model.add(tensorflow.keras.layers.Dense(2048, activation='relu'))
        model.add(tensorflow.keras.layers.Dropout(0.25))
        model.add(tensorflow.keras.layers.Dense(self.nb_classes, activation='softmax'))
        #model = tensorflow.keras.utils.multi_gpu_model(model, gpus=8)
        model.compile(
            loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
        stop = tensorflow.keras.callbacks.EarlyStopping(
            monitor='val_acc', min_delta=0.001, patience=2, mode='auto')
        model.fit(
            self.X_train, self.y_train, batch_size=batch_size, epochs=nb_epoch,
            verbose=1, validation_split=0.2, callbacks=[stop])
        return model

    def reshape_data(self, img_rows, img_cols, channels, nb_classes):
        """
        Reshapes arrays into format for MXNet.

        Parameters
            img_rows: image array height
            img_cols: image array width
            channels: specify if image is grayscale (1) or RGB (3)
            nb_classes: number of image classes/categories

        Return
            none
        """
        self.X_train = self.X_train.reshape(
            self.X_train.shape[0], img_rows, img_cols, channels)
        self.X_train = self.X_train.astype('float32')
        self.X_train /= 255
        self.y_train = tensorflow.keras.utils.to_categorical(
            self.y_train, self.nb_classes)
        self.X_test = self.X_test.reshape
        (2000, img_rows, img_cols, channels)
        #self.X_test = self.X_test.astype('float32')
        self.X_test /= 255
        self.y_test = tensorflow.keras.utils.to_categorical(
            self.y_test, self.nb_classes)
        print('X_train shape:', self.X_train.shape)
        print('X_test shape:', self.X_test.shape)
        print('y_train shape:', self.y_train.shape)
        print('y_test shape:', self.y_test.shape)

    def save_model(self, score, model_name):
        """
        Saves the model, based on scoring criteria input.

        Parameters
            score: scoring metric used to save model
            model_name: name for the model to be saved

        Return
            none
        """
        if score >= 0.75:
            print('saving model')
            self.model.save(model_name + '_recall_' + str(round(score, 4)) + '.h5')
        else:
            print('model not saved. score:', score)

    def split_data(self, y_file_path, X, test_size=0.2):
        """
        Split data into test and training data sets.

        Parameters
            y_file_path: path to CSV containing labels
            X: NumPy array of arrays
            test_data_size: size of test/train split. Value from 0 to 1
        
        Return
            none
        """
        labels = pandas.read_csv(y_file_path)
        labels = labels[:10000]
        X = cupy.load(X)
        y = cupy.array(labels['level'])
        self.weights = sklearn.utils.class_weight.compute_class_weight(
            'balanced', classes=numpy.unique(y), y=y)
        self.X_train, self.X_test, self.y_train, self.y_test = \
            sklearn.model_selection.train_test_split(
                X, y, test_size=test_size, random_state=42)

In [ ]:
EyeNet()